# Monitoring Data Drift

Ce notebook analyse la dérive des données (data drift) entre :

- Les distributions de référence (données d'entrainement, issues de la table `ref_feature_dist`)
- Les données observées en production (issues de la table `prod_requests`)

La métrique utilisée est le **Population Stability Index (PSI)**.

> **Remarque :** Le seuil de drift utilisé est paramétrable (voir variable `drift_threshold` dans le code).

## Import et config

In [ ]:

from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# --- Robust project root detection (remonte jusqu'à trouver "core" et "monitoring")
HERE = Path.cwd()

def find_project_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "core").exists() and (p / "monitoring").exists():
            return p
    return start

PROJECT_ROOT = find_project_root(HERE)

sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / "monitoring"))

In [2]:
from core.db.conn import init_db
from monitoring.lib.constants import DEFAULTS
from monitoring.lib.security import EXCLUDED_FEATURES
from monitoring.lib.data import load_prod_data, load_reference
from monitoring.lib.ops import latency_stats_ms, error_rate, success_rate
from monitoring.lib.timings import extract_timings, compute_timing_stats
from monitoring.lib.drift import compute_drift_table, count_drift
from monitoring.lib.charts import (
    hist_latency,
    bar_status_codes,
    pie_decisions,
    bar_top_drift,
    bar_ref_vs_prod,
)


REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Init DB (idempotent)
init_db()
print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = c:\Users\yoann\Documents\open classrooms\projet 8\livrables v2\pret a dépenser


In [4]:
endpoint = DEFAULTS.get("endpoint", "/predict")
limit = int(DEFAULTS.get("limit", 0))
time_window = DEFAULTS.get("time_window", "all")
drift_threshold = float(DEFAULTS.get("drift_threshold", 0.25))

limit_val = None if limit == 0 else limit

print("endpoint =", endpoint)
print("limit_val =", limit_val)
print("time_window =", time_window)
print("drift_threshold =", drift_threshold)

endpoint = /predict
limit_val = None
time_window = all
drift_threshold = 0.25


In [5]:
prod_meta, prod_inputs, prod_outputs, rows = load_prod_data(
    endpoint=endpoint,
    limit=limit_val,
    time_window=time_window,
    excluded_features=EXCLUDED_FEATURES,
)

print("prod_meta:", prod_meta.shape)
print("prod_inputs:", prod_inputs.shape)
print("prod_outputs:", prod_outputs.shape)

assert not prod_meta.empty, "Aucune requête trouvée en DB (prod_requests)."

prod_meta: (2000, 6)
prod_inputs: (2000, 125)
prod_outputs: (2000, 5)


## Santé

In [6]:
lat_total = pd.to_numeric(prod_meta["latency_ms"], errors="coerce")
stats_total = latency_stats_ms(lat_total)

print("\n=== OPS ===")
print("Nb requêtes:", len(prod_meta))
print("Succès (200) %:", round(success_rate(prod_meta["status_code"]), 2))
print("Erreurs (>=400) %:", round(error_rate(prod_meta["status_code"]), 2))
print(
    "p50/p95/p99 total (ms):",
    round(stats_total["p50"], 2),
    round(stats_total["p95"], 2),
    round(stats_total["p99"], 2),
)


=== OPS ===
Nb requêtes: 2000
Succès (200) %: 100.0
Erreurs (>=400) %: 0.0
p50/p95/p99 total (ms): 2.85 3.97 4.76


In [7]:
# Graphes ops
fig_lat = hist_latency(lat_total, "Distribution latence totale (ms)")
fig_lat.show()

status_counts = prod_meta["status_code"].value_counts(dropna=False).sort_index()
fig_status = bar_status_codes(status_counts, "Codes HTTP")
fig_status.show()


timing_df = extract_timings(prod_outputs)
timing_stats = compute_timing_stats(timing_df)

## Détail latence

In [8]:





print("\n=== TIMINGS ===")
if not timing_stats:
    print("Aucun champ timing trouvé dans outputs.")
else:
    for k in ["db_ms", "validation_ms", "inference_ms", "total_ms"]:
        if k in timing_stats:
            s = timing_stats[k]
            print(f"{k}: p50={s['p50']:.2f} p95={s['p95']:.2f} p99={s['p99']:.2f} mean={s['mean']:.2f}")

    # Graphes timing
    for col, title in [
        ("db_ms", "DB time (ms)"),
        ("validation_ms", "Validation time (ms)"),
        ("inference_ms", "Inference time (ms)"),
        ("total_ms", "Total timing (ms)"),
    ]:
        if col in timing_df.columns:
            hist_latency(timing_df[col], title).show()





=== TIMINGS ===
db_ms: p50=1.73 p95=2.64 p99=3.24 mean=1.70
validation_ms: p50=0.00 p95=0.63 p99=1.01 mean=0.23
inference_ms: p50=1.03 p95=1.57 p99=2.02 mean=0.88
total_ms: p50=2.85 p95=3.97 p99=4.76 mean=2.83


## Data Drift (PSI) — référence en DB

Nous allons maintenant analyser la stabilité des distributions de variables entre la référence et la production à l'aide du PSI. Le seuil critique est fixé à 0.25, mais il peut être ajusté selon les besoins de surveillance.

In [9]:
ref_rows = load_reference()
assert ref_rows, "Aucune référence en DB (ref_feature_dist)."

drift_df = compute_drift_table(
    prod_inputs=prod_inputs,
    ref_rows=ref_rows,
    excluded_features=EXCLUDED_FEATURES,
)

def risk_level(psi: float) -> str:
    if pd.isna(psi):
        return "NA"
    if psi < 0.1:
        return "Stable"
    if psi < 0.25:
        return "Moderate"
    return "High Drift"

drift_df["risk_level"] = drift_df["psi"].apply(risk_level)

n_drift = count_drift(drift_df, threshold=drift_threshold)

print("\n=== DRIFT PSI ===")
print(f"Nb features PSI > {drift_threshold} :", n_drift)

display(drift_df.head(20))


out_csv = REPORTS_DIR / "drift_psi_results.csv"
drift_df.to_csv(out_csv, index=False)
print("Exporté:", out_csv)






=== DRIFT PSI ===
Nb features PSI > 0.25 : 0


,feature,psi,type,risk_level
32,NAME_CONTRACT_TYPE,0.239409,categorical,Moderate
15,AMT_GOODS_PRICE,0.105578,numeric,Moderate
6,AMT_CREDIT,0.076154,numeric,Stable
37,DAYS_ID_PUBLISH,0.071889,numeric,Stable
83,BUREAU_DAYS_CREDIT_MAX,0.053147,numeric,Stable
73,BUREAU_AMT_CREDIT_MAX_OVERDUE_STD,0.050581,numeric,Stable
68,PREV_CC_CNT_DRAWINGS_ATM_CURRENT_MEAN_MIN,0.046483,numeric,Stable
65,PREV_CC_CNT_DRAWINGS_ATM_CURRENT_MEAN_MEAN,0.045454,numeric,Stable
35,PREV_CC_CNT_DRAWINGS_ATM_CURRENT_MEAN_MAX,0.045042,numeric,Stable
33,PREV_CC_CC_UTILIZATION_MAX_MAX,0.044494,numeric,Stable


Exporté: c:\Users\yoann\Documents\open classrooms\projet 8\livrables v2\pret a dépenser\reports\drift_psi_results.csv


In [11]:
TOPK = 10
topk_df = drift_df.dropna(subset=["psi"]).head(TOPK).copy()
fig = bar_top_drift(topk_df, f"Top {TOPK} Drift Features (PSI)")
fig.show()

In [13]:
import numpy as np

feature_name = "NAME_CONTRACT_TYPE"

# Récupération référence
ref_one = [r for r in ref_rows if r["feature"] == feature_name][0]
ref_dist = ref_one.get("ref_dist_json") or {}
labels_ref = ref_dist.get("labels") or []
ref_p = np.array(ref_dist.get("p") or [], dtype=float)

# Production
prod_s = prod_inputs[feature_name]

from monitoring.lib.drift import prod_dist_categorical, psi_from_dists

_, prod_p = prod_dist_categorical(prod_s, labels_ref=labels_ref)

# DataFrame pour plot
df_plot = pd.DataFrame({
    "label": labels_ref,
    "ref": ref_p,
    "prod": prod_p
})

from monitoring.lib.charts import bar_ref_vs_prod

fig_detail = bar_ref_vs_prod(df_plot, feature_name)
fig_detail.show()

# PSI affiché
psi_val = psi_from_dists(ref_p, prod_p)
print(f"PSI({feature_name}) = {psi_val:.4f}")

PSI(NAME_CONTRACT_TYPE) = 0.2394


#### Zoom sur la variable présentant le PSI le plus élevé

Nous détaillons ici la variable ayant le PSI maximum afin de mieux comprendre la nature de la dérive observée.

### Résultats principaux

**L’analyse du drift** a été réalisée sur les **125 features** réellement utilisées par le modèle en production (*issues de prod_requests.inputs*).

- **PSI maximum** : 0.239 — NAME_CONTRACT_TYPE (catégorielle)
- **Niveau** : **Drift modéré** (proche du seuil critique 0.25)
- 1 variable en zone modérée significative
- Toutes les autres variables sont **stables (< 0.1)** ou très faiblement impactées

### Interprétation

Nous allons maintenant interpréter les résultats obtenus et évaluer leur impact potentiel sur la performance du modèle.

La distribution des types de contrat en production diffère légèrement de celle observée à l’entraînement.
Cependant :
- Aucun drift fort (> 0.25)
- Aucune dérive systémique globale
- Population globalement stable

### Conclusion

L’analyse PSI ne détecte pas tous les types de dérive (ex : changement de corrélation, évolution lente non détectée). Une analyse complémentaire pourrait être envisagée pour une surveillance plus fine.

Le modèle opère sur une population proche de celle d'entraînement.
Une surveillance continue est maintenue sur NAME_CONTRACT_TYPE, sans nécessité d’action corrective immédiate.